In [1]:
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

In [2]:
PROJECT_ROOT=Path.cwd().parent
# Khai báo đường dẫn
DATA_DIR=PROJECT_ROOT/"data"/"raw"

PARQUET_PATH=DATA_DIR/"yellow_tripdata_2026-01.parquet"
LOOKUP_PATH=DATA_DIR/"taxi_zone_lookup.csv"

# Kiểm tra file tồn tại
print(DATA_DIR)
print("Project root: ",PROJECT_ROOT)
print("Parquet exists: ",PARQUET_PATH.exists())
print("Lookup exists: ",LOOKUP_PATH.exists())

c:\Users\Acer\Documents\DATA ANALYST\Portfolio\NYC Taxi Project\data\raw
Project root:  c:\Users\Acer\Documents\DATA ANALYST\Portfolio\NYC Taxi Project
Parquet exists:  True
Lookup exists:  True


In [3]:
df=pd.read_parquet(PARQUET_PATH)
zones=pd.read_csv(LOOKUP_PATH)

display(df.head(10))
display(zones.head(10))

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
5,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,N,144,137,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.0,0.75
7,1,2026-01-01 00:34:28,2026-01-01 00:59:05,0.0,2.90,1.0,N,50,234,1,22.6,4.25,0.5,5.65,0.0,1.0,34.00,2.5,0.0,0.75
8,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,N,161,45,1,37.3,1.00,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75
9,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3.0,1.83,1.0,N,237,263,1,10.7,1.00,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
5,6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
6,7,Queens,Astoria,Boro Zone
7,8,Queens,Astoria Park,Boro Zone
8,9,Queens,Auburndale,Boro Zone
9,10,Queens,Baisley Park,Boro Zone


In [12]:
# Kiểm tra giá trị null
showDFnull=df.isnull().sum()
showZNnull=zones.isnull().sum()
print(f"Giá trị null \n{showDFnull[showDFnull>0]}\n")
print(f"Giá trị Zones null \n{showZNnull[showZNnull>0]}")


Giá trị null 
passenger_count         1088058
RatecodeID              1088058
store_and_fwd_flag      1088058
congestion_surcharge    1088058
Airport_fee             1088058
dtype: int64

Giá trị Zones null 
Borough         1
Zone            1
service_zone    2
dtype: int64


In [11]:

# Kiểm tra NULL trong bảng tra cứu khu vực
zones_null = zones[zones.isnull().any(axis=1)]

print("Số dòng có NULL trong bảng zones:", len(zones_null))
display(zones_null)


Số dòng có NULL trong bảng zones: 2


,LocationID,Borough,Zone,service_zone
263,264,Unknown,NaN,NaN
264,265,NaN,Outside of NYC,NaN


In [ ]:
colNull= [
    'passenger_count',
    'RatecodeID',
    'store_and_fwd_flag',
    'congestion_surcharge',
    'Airport_fee'
]
null_per_row = df[colNull].isnull().sum(axis=1)


In [18]:
# Kiểm tra các cột chứa các giá trị null
df_null_group= df.loc[null_per_row==5]

display(df_null_group[
        [
            "VendorID",
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "trip_distance",
            "fare_amount",
            "total_amount"
        ]
    ].head(10))


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,total_amount
2636831,2,2026-01-01 00:28:47,2026-01-01 00:49:30,10.58,82.01,89.19
2636832,2,2026-01-01 00:54:24,2026-01-01 01:15:01,11.04,51.77,53.27
2636833,2,2026-01-01 00:12:31,2026-01-01 00:41:02,4.25,32.17,36.92
2636834,2,2026-01-01 00:48:56,2026-01-01 01:08:00,2.05,18.29,24.04
2636835,2,2026-01-01 00:32:09,2026-01-01 00:41:05,0.97,15.02,22.77
2636836,2,2026-01-01 00:24:17,2026-01-01 00:31:51,1.99,13.87,25.37
2636837,2,2026-01-01 00:49:31,2026-01-01 01:21:03,6.44,24.77,29.52
2636838,2,2026-01-01 00:39:35,2026-01-01 01:27:01,7.40,59.36,76.93
2636839,2,2026-01-01 00:27:04,2026-01-01 00:36:03,0.98,9.87,14.62
2636840,2,2026-01-01 00:44:00,2026-01-01 01:08:00,4.30,33.59,44.09


In [30]:
df_result=df_null_group.groupby('VendorID')['trip_distance'].count().reset_index(name='count')
display(df_result)

,VendorID,count
0,1,139406
1,2,944635
2,6,4017


In [20]:
df_null_group = df.loc[null_per_row == 5]
df_complete_group = df.loc[null_per_row == 0]

In [24]:
# So sánh Mean và Median của 2 nhóm Null và Not Null
compare_data = pd.DataFrame({
    ("Nhom Null", "Mean"): df_null_group[
        ['trip_distance', 'fare_amount', 'total_amount']
    ].mean(),

    ("Nhom Null", "Median"): df_null_group[
        ['trip_distance', 'fare_amount', 'total_amount']
    ].median(),

    ("Nhom Not Null", "Mean"): df_complete_group[
        ['trip_distance', 'fare_amount', 'total_amount']
    ].mean(),

    ("Nhom Not Null", "Median"): df_complete_group[
        ['trip_distance', 'fare_amount', 'total_amount']
    ].median()
})

display(compare_data.round(2))

Nhom Null        Nhom Not Null       
                   Mean Median          Mean Median
trip_distance     13.75   2.40          3.45   1.63
fare_amount       25.38  22.21         18.92  12.80
total_amount      31.68  28.08         28.14  21.06

### Kiểm tra giá trị Null cho thấy: Các chuyến đi của tập trung vào Vendor 1,2,6

- Có 1.088.058 bản ghi bị NULL đồng thời ở 5 cột passenger_count, RatecodeID, store_and_fwd_flag, congestion_surcharge và Airport_fee.

- Khi so sánh với nhóm không có NULL ở 5 cột này, nhóm NULL có quãng đường và tiền cước trung bình cao hơn. Đáng chú ý, Mean của trip_distance là 13,75 miles, trong khi Median chỉ là 2,40 miles cho thấy phân phối có thể bị ảnh hưởng bởi một số giá trị lớn.

- Kết quả cho thấy hai nhóm có đặc điểm khác nhau. Tuy nhiên, chưa đủ cơ sở để kết luận các bản ghi NULL là dữ liệu lỗi hoặc cần xóa. Cần kiểm tra thêm các giá trị bất thường và xác định ý nghĩa của từng cột trước khi quyết định phương án xử lý.